In [1]:
# =============================================================================
# Running all the cells in this notebook will produce the datasets of SPDNN
# from the model used in Ma et al.
# It will not generate exactly the same dataset given its stochasticity.
# =============================================================================
# Standard library
import os
import sys
from pathlib import Path

# Numerical computing
import numpy as np

# Data handling
from tqdm import tqdm
import joblib

# Machine Learning
import torch
from torchvision import datasets, transforms


# Project imports
def _find_repo_root(markers=("config.py", "CITATION.cff")):
    """
    Locate the repository root by walking up from the current working
    directory until a directory containing all `markers` is found.

    More robust than `Path.cwd().parent[.parent]`: independent of which
    directory the Jupyter kernel happened to start in, and fails loudly
    (rather than silently importing the wrong `config.py`) if the notebook
    is run outside the repository.
    """
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(
        f"Could not locate repository root from {start}. "
        f"Expected to find {markers} in some parent directory. "
        f"Open this notebook from inside the optical-eigentask-learning "
        f"repository."
    )


REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "code"))

from config import (
    DATA_DIR,
    SPDNN_DIR,
)

# Change this parentPath to the Path you download and decompress the zip file from
# https://zenodo.org/records/8188270
# Acknowledgment to Ma et al.
parentPath = DATA_DIR / "code/SPDNN_data"
dataPath = parentPath / "data"
srcPath = parentPath / "src"
modelPath = parentPath / "models"
savePath = parentPath / "results"

sys.path.append(srcPath.as_posix())

In [ ]:
train_batch_size = 100
test_batch_size = 100
image_size = 28

MNISTPath = DATA_DIR

# Dataset
testset = datasets.MNIST(
    MNISTPath,
    train=False,
    transform=transforms.Compose(
        [transforms.Resize(image_size), transforms.ToTensor()]
    ),
)
test_loader = torch.utils.data.DataLoader(
    testset, batch_size=test_batch_size, shuffle=False
)

trainset = datasets.MNIST(
    MNISTPath,
    train=True,
    transform=transforms.Compose(
        [transforms.Resize(image_size), transforms.ToTensor()]
    ),
)
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=train_batch_size, shuffle=False
)

for i, (data, target) in enumerate(test_loader):
    test_input_i, test_labels_i = data, target
    break

In [ ]:
from SPDNN_MLP import PDMLP_1 as MLP


def renorm(data, min_val, max_val):
    return data * (max_val - min_val) + min_val


device = "cpu"

exp_models = {}
exp_Ns = []
for f in os.listdir(modelPath):
    if f.startswith(f"MNIST{image_size}x{image_size}_exp"):
        N = int(f.split("_")[2][1:].split(".")[0])
        exp_Ns.append(N)
        print(N, f)

        model = MLP(n_hidden=N, n_input=image_size**2, n_output=10)

        model.load_state_dict(torch.load(modelPath / f, map_location=device))
        model.to(device)
        model.eval()
        exp_models[N] = model

exp_Ns.sort()
exp_Ns

50 MNIST28x28_exp_N50.pth
400 MNIST28x28_exp_N400.pth
200 MNIST28x28_exp_N200.pth
300 MNIST28x28_exp_N300.pth
100 MNIST28x28_exp_N100.pth


/var/folders/kh/_3wcw6gd5mzdh894tk231qtr0000gn/T/ipykernel_11008/3737109468.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(modelPath /

[50, 100, 200, 300, 400]

In [ ]:
num_shots = 30
acc = {50: 0.8727, 100: 0.9283, 200: 0.9463, 300: 0.9603, 400: 0.9623}

for N in acc:
    train_shots, train_labels = np.zeros((60000, num_shots, N)), np.zeros(60000)
    test_shots, test_labels = np.zeros((10000, num_shots, N)), np.zeros(10000)

    for i, (data, target) in tqdm(enumerate(train_loader)):
        shots = np.transpose(
            np.array(
                [
                    exp_models[N]
                    .act(
                        exp_models[N].fc1(
                            renorm(data, 0.02, 1).reshape(train_batch_size, -1)
                        )
                    )
                    .detach()
                    .numpy()
                    for _ in range(num_shots)
                ]
            ),
            [1, 0, 2],
        )
        train_shots[i * train_batch_size : (i + 1) * train_batch_size] = shots
        train_labels[i * train_batch_size : (i + 1) * train_batch_size] = target

    for i, (data, target) in tqdm(enumerate(test_loader)):
        shots = np.transpose(
            np.array(
                [
                    exp_models[N]
                    .act(
                        exp_models[N].fc1(
                            renorm(data, 0.02, 1).reshape(test_batch_size, -1)
                        )
                    )
                    .detach()
                    .numpy()
                    for _ in range(num_shots)
                ]
            ),
            [1, 0, 2],
        )
        test_shots[i * test_batch_size : (i + 1) * test_batch_size] = shots
        test_labels[i * test_batch_size : (i + 1) * test_batch_size] = target

    expt_shots = np.transpose(
        np.array(
            joblib.load(dataPath / f"actvs/actv_N{N}_{acc[N]:.4f}.pkl"), dtype=int
        ),
        [1, 0, 2],
    )
    expt_labels = test_labels_i

    dataset_path = SPDNN_DIR
    np.savez_compressed(
        dataset_path / f"mnist_spdnn_N{N}_bool.npz",
        train_shots=train_shots.astype(bool),
        train_labels=train_labels.astype(int),
        test_shots=test_shots.astype(bool),
        test_labels=test_labels.astype(int),
        expt_shots=expt_shots.astype(bool),
        expt_labels=expt_labels.numpy().astype(int),
    )

0it [00:00, ?it/s]

/var/folders/kh/_3wcw6gd5mzdh894tk231qtr0000gn/T/ipykernel_11008/1917752483.py:26: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_labels[i * train_batch_size : (i + 1) * train_batch_size] = target
600it [00:06, 90.94it/s]
0it [00:00, ?it/s]/var/folders/kh/_3wcw6gd5mzdh894tk231qtr0000gn/T/ipykernel_11008/1917752483.py:46: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  test_labels[i * test_batch_size : (i + 1) * test_batch_size] = target
100it [00:01, 95.73it/s]
600it [00